# NB02: Hybrid models (M1–M5)

Trains models M1–M5 with nested spatial CV and computes bootstrap ΔRMSE for H1 and H2.

**Outputs**
- `data/cv_results.csv` — per-model per-target RMSE
- `data/bootstrap_delta_rmse.csv` — bootstrap CI for H1 and H2
- `data/oof_predictions.parquet` — out-of-fold predictions for all models and targets

**Run on**: JupyterHub (NB00 must be run first)

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

for _cand in [Path.cwd() / 'scripts', Path.cwd().parent / 'scripts']:
    if _cand.exists():
        sys.path.insert(0, str(_cand))
        break

DATA_DIR = next(p for p in [Path.cwd() / 'data', Path.cwd().parent / 'data'] if p.exists())
FIGURES_DIR = next(p for p in [Path.cwd() / 'figures', Path.cwd().parent / 'figures'] if p.exists())
FIGURES_DIR.mkdir(exist_ok=True)

feature_matrix = pd.read_parquet(DATA_DIR / 'feature_matrix.parquet')
spatial_blocks = pd.read_csv(DATA_DIR / 'spatial_blocks.csv').set_index('sample_id')
block_labels = spatial_blocks.loc[feature_matrix.index, 'block'].values

targets = ['log_Cu_ppm', 'log_Zn_ppm', 'log_Pb_ppm', 'log_Ni_ppm']
targets_avail = [t for t in targets if t in feature_matrix.columns]

print(f'Feature matrix: {feature_matrix.shape}')
print(f'Block distribution: {dict(zip(*np.unique(block_labels, return_counts=True)))}')

## 1. Nested spatial CV for all models

In [ ]:
from modelling import nested_spatial_cv, make_results_table

MODEL_CONFIGS = [
    ('M1', 'ridge'),
    ('M2', 'xgboost'),
    ('M3', 'xgboost'),
    ('M4', 'xgboost'),
]

all_cv_results = []
oof_preds_dict = {}

for target in targets_avail:
    y = feature_matrix[target]
    for model_name, model_type in MODEL_CONFIGS:
        print(f'  Fitting {model_name} for {target}...')
        result = nested_spatial_cv(
            feature_df=feature_matrix,
            target_series=y,
            block_labels=block_labels,
            model_name=model_name,
            model_type=model_type,
        )
        all_cv_results.append({
            'model': model_name,
            'target': target,
            'overall_rmse': result['overall_rmse'],
            'fold_rmse_mean': np.mean(result['fold_rmse']),
            'fold_rmse_sd': np.std(result['fold_rmse']),
        })
        oof_preds_dict[f'{model_name}_{target}'] = result['oof_preds']

cv_df = pd.DataFrame(all_cv_results)
cv_df.to_csv(DATA_DIR / 'cv_results.csv', index=False)
print(cv_df.pivot(index='model', columns='target', values='overall_rmse').round(4))

In [ ]:
# M5: Multi-output — train jointly on all available targets
from modelling import build_xgboost, get_features, _drop_nan_rows, rmse
from spatial_utils import spatial_cv_splits

try:
    from xgboost import XGBRegressor
    X_m5 = get_features(feature_matrix, 'M5')
    Y_m5 = feature_matrix[targets_avail]
    splits_m5 = spatial_cv_splits(block_labels)
    oof_m5 = pd.DataFrame(np.nan, index=feature_matrix.index, columns=targets_avail)

    for train_idx, test_idx in splits_m5:
        Xtr = X_m5.iloc[train_idx]
        Ytr = Y_m5.iloc[train_idx]
        Xte = X_m5.iloc[test_idx]
        valid_rows = ~Xtr.isna().any(axis=1) & Ytr.notna().all(axis=1)
        Xtr_c, Ytr_c = Xtr[valid_rows], Ytr[valid_rows]

        te_valid = ~Xte.isna().any(axis=1)
        Xte_c = Xte[te_valid]

        models = {}
        for col in targets_avail:
            m = build_xgboost()
            m.fit(Xtr_c, Ytr_c[col])
            models[col] = m

        for col, m in models.items():
            oof_m5.loc[Xte_c.index, col] = m.predict(Xte_c)

    for col in targets_avail:
        valid = oof_m5[col].notna() & Y_m5[col].notna()
        err = rmse(Y_m5.loc[valid, col].values, oof_m5.loc[valid, col].values)
        all_cv_results.append({'model': 'M5', 'target': col, 'overall_rmse': err,
                                'fold_rmse_mean': np.nan, 'fold_rmse_sd': np.nan})
        oof_preds_dict[f'M5_{col}'] = oof_m5[col]
    print('M5 complete')
except Exception as e:
    print(f'M5 failed: {e}')

In [ ]:
# Save OOF predictions
oof_df = pd.DataFrame(oof_preds_dict, index=feature_matrix.index)
oof_df.to_parquet(DATA_DIR / 'oof_predictions.parquet')

cv_df = pd.DataFrame(all_cv_results)
cv_df.to_csv(DATA_DIR / 'cv_results.csv', index=False)
print('Saved cv_results.csv and oof_predictions.parquet')

## 2. Hypothesis tests H1 and H2 (bootstrap ΔRMSE)

In [ ]:
from modelling import bootstrap_delta_rmse, build_ridge, _drop_nan_rows, ENV_FEATURES
from spatial_utils import spatial_cv_splits

boot_records = []

for target in targets_avail:
    y_series = feature_matrix[target]
    y_vals = y_series.values

    # Re-run B1 (pH only ridge) to get OOF predictions for bootstrap
    X_b1 = feature_matrix[['ph']]
    oof_b1 = np.full(len(y_series), np.nan)
    splits_boot = spatial_cv_splits(block_labels)
    for train_idx, test_idx in splits_boot:
        Xtr, ytr = _drop_nan_rows(X_b1.iloc[train_idx], y_series.iloc[train_idx])
        # Track which test rows are non-NaN (not just the first N)
        nan_mask = X_b1.iloc[test_idx].isna().any(axis=1) | y_series.iloc[test_idx].isna()
        valid_test_pos = np.array(test_idx)[~nan_mask.values]
        Xte = X_b1.iloc[test_idx][~nan_mask]
        if len(Xtr) > 5 and len(Xte) > 0:
            m = build_ridge().fit(Xtr, ytr)
            oof_b1[valid_test_pos] = m.predict(Xte)

    oof_m1 = oof_preds_dict.get(f'M1_{target}')
    oof_m2 = oof_preds_dict.get(f'M2_{target}')
    oof_m4 = oof_preds_dict.get(f'M4_{target}')

    # H1: B1 vs M1 (positive delta = B1 worse = CWM helps)
    if oof_m1 is not None:
        common = y_series.notna().values & oof_m1.notna().values & ~np.isnan(oof_b1)
        if common.sum() > 10:
            h1 = bootstrap_delta_rmse(
                y_vals[common], oof_b1[common], oof_m1.values[common]
            )
            h1.update({'hypothesis': 'H1', 'target': target,
                       'comparison': 'B1_minus_M1', 'model_A': 'B1', 'model_B': 'M1'})
            boot_records.append(h1)

    # H2: M4 vs M2 (positive delta = M4 worse = CWM helps beyond env)
    if oof_m4 is not None and oof_m2 is not None:
        common = y_series.notna().values & oof_m4.notna().values & oof_m2.notna().values
        if common.sum() > 10:
            h2 = bootstrap_delta_rmse(
                y_vals[common], oof_m4.values[common], oof_m2.values[common]
            )
            h2.update({'hypothesis': 'H2', 'target': target,
                       'comparison': 'M4_minus_M2', 'model_A': 'M4', 'model_B': 'M2'})
            boot_records.append(h2)

boot_df = pd.DataFrame(boot_records)
boot_df.to_csv(DATA_DIR / 'bootstrap_delta_rmse.csv', index=False)
print(boot_df[['hypothesis', 'target', 'observed_delta_rmse', 'boot_ci_lo', 'boot_ci_hi']].to_string(index=False))

## 3. Conformal prediction intervals

In [ ]:
from modelling import ConformalPredictor, get_features, build_xgboost, _drop_nan_rows
from spatial_utils import spatial_cv_splits

# Fit conformal predictor using first block as holdout, second as calibration
target = targets_avail[0]
print(f'Conformal prediction example for {target}')

splits_cp = spatial_cv_splits(block_labels)
train_idx, cal_idx = splits_cp[0]  # train on non-block-0 samples

X_m2 = get_features(feature_matrix, 'M2')
y_ser = feature_matrix[target]

Xtr, ytr = _drop_nan_rows(X_m2.iloc[train_idx], y_ser.iloc[train_idx])
Xcal, ycal = _drop_nan_rows(X_m2.iloc[cal_idx], y_ser.iloc[cal_idx])

model_cp = build_xgboost().fit(Xtr, ytr)
cp = ConformalPredictor(alpha=0.10)
cp.calibrate(model_cp, Xcal, ycal)
lo, hi = cp.predict_interval(Xcal)
coverage = ((ycal.values >= lo) & (ycal.values <= hi)).mean()
print(f'Empirical coverage on calibration set: {coverage:.3f} (target: 0.90)')

## 4. Prediction scatter plots

In [ ]:
from evaluation import plot_prediction_scatter

for target in targets_avail:
    y = feature_matrix[target]
    oof_m2 = oof_preds_dict.get(f'M2_{target}')
    if oof_m2 is not None:
        plot_prediction_scatter(
            y, oof_m2.values,
            target_name=target.replace('log_', ''),
            save_path=FIGURES_DIR / f'scatter_M2_{target}.png',
        )
print('Scatter plots saved.')

## 5. H5 — Geographic vs random CV degradation

Tests whether CWM-rich models degrade more under spatial block CV than env-only models. Block RMSE from Section 1; random RMSE from KFold(n_splits=5, shuffle=True, random_state=42).

In [ ]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold

for _cand in [Path.cwd() / 'scripts', Path.cwd().parent / 'scripts']:
    if _cand.exists():
        sys.path.insert(0, str(_cand))
        break
DATA_DIR = next(p for p in [Path.cwd() / 'data', Path.cwd().parent / 'data'] if p.exists())

from modelling import get_features, build_xgboost, _drop_nan_rows, rmse

feature_matrix = pd.read_parquet(DATA_DIR / 'feature_matrix.parquet')
block_cv_df = pd.read_csv(DATA_DIR / 'cv_results.csv')
targets = ['log_Cu_ppm', 'log_Zn_ppm', 'log_Pb_ppm', 'log_Ni_ppm']
targets_avail = [t for t in targets if t in feature_matrix.columns]

kf = KFold(n_splits=5, shuffle=True, random_state=42)
h5_records = []

for target in targets_avail:
    y = feature_matrix[target]
    for model_name in ['M2', 'M4']:
        X = get_features(feature_matrix, model_name)
        # Block CV RMSE (already computed in Section 1)
        block_rmse = block_cv_df.loc[
            (block_cv_df.model == model_name) & (block_cv_df.target == target),
            'overall_rmse'
        ].values[0]

        # Random KFold CV
        oof_random = np.full(len(y), np.nan)
        row_idx = np.arange(len(y))
        for train_pos, test_pos in kf.split(row_idx):
            Xtr, ytr = _drop_nan_rows(X.iloc[train_pos], y.iloc[train_pos])
            te_valid = ~X.iloc[test_pos].isna().any(axis=1) & ~y.iloc[test_pos].isna()
            valid_test_pos = test_pos[te_valid.values]
            Xte = X.iloc[test_pos][te_valid]
            if len(Xtr) > 5 and len(Xte) > 0:
                m = build_xgboost().fit(Xtr, ytr)
                oof_random[valid_test_pos] = m.predict(Xte)

        valid = y.notna().values & ~np.isnan(oof_random)
        rand_rmse = rmse(y.values[valid], oof_random[valid])

        h5_records.append({
            'model': model_name, 'target': target,
            'block_rmse': block_rmse,
            'random_rmse': rand_rmse,
            'degradation_ratio': block_rmse / rand_rmse,
        })
        print(f'{model_name} {target}: block={block_rmse:.4f}  random={rand_rmse:.4f}  ratio={block_rmse/rand_rmse:.3f}')

h5_df = pd.DataFrame(h5_records)
h5_df.attrs = {}
h5_df.to_csv(DATA_DIR / 'h5_degradation_ratios.csv', index=False)
print('\nDegradation ratio (block/random) — higher = more geographic sensitivity:')
print(h5_df.pivot_table(index='model', columns='target', values='degradation_ratio').round(3))
print('\nH5: M2 degrades MORE than M4? (ratio_M2 > ratio_M4)')
for target in targets_avail:
    r_m2 = h5_df.loc[(h5_df.model == 'M2') & (h5_df.target == target), 'degradation_ratio'].values[0]
    r_m4 = h5_df.loc[(h5_df.model == 'M4') & (h5_df.target == target), 'degradation_ratio'].values[0]
    print(f'  {target}: M2={r_m2:.3f}  M4={r_m4:.3f}  M2>M4? {r_m2 > r_m4}')